In [1]:
# import libraries
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
#Project Path
PROJECT_ROOT = Path("modules")
print("Project root:", PROJECT_ROOT)

Project root: modules


In [3]:
# csv path
TRACKING_RUNS_PATH = PROJECT_ROOT / "Tracking" / "runs" / "track_lkpg"
HOMOGRAPHY_MILY_RUNS_PATH = PROJECT_ROOT / "Homography" / "runs" / "track_lkpg_mily"
HOMOGRAPHY_RUNS_PATH = PROJECT_ROOT / "Homography" / "runs" / "track_lkpg"

In [8]:
# Fix to display a clickable window in the Jupyter Notebook when using matplotlib
%matplotlib qt

In [10]:
"""
select_points.py

Interactively select N corresponding points (default 5) on:
  1) A georeferenced orthophoto (GeoTIFF or any rasterio-readable raster)
  2) A plain PNG image of the same scene, taken from a different angle

Output:
  - Orthophoto points  -> real-world coordinates in WGS84 (lon, lat)
  - PNG points         -> pixel coordinates (x, y)

Results are written to a tab-separated text file, one row per point pair,
so point i on the orthophoto corresponds to point i on the PNG.

Requirements:
    pip install rasterio pyproj matplotlib pillow numpy

Usage:
    python select_points.py --ortho orthophoto.tif --png photo.png --out points.txt --n 5

Instructions when running:
    - A window opens showing the orthophoto. Click the points in the exact
      order you want them recorded. The window closes automatically after
      the last click.
    - A second window opens showing the PNG. Click the corresponding points
      in the SAME order (point 1 on the PNG must be the same real-world
      feature as point 1 on the orthophoto, etc.).
"""
ORTHOPHOTO_PATH = "../../test_ortofoto_lkpg/647_53_5050_2023_clipped.tif"
FRAME_PATH = "../../output/frame_0.png"
OUTPUT_TXT_PATH = "../../output/ref_points.txt"
NB_REF_POINTS = 5

import sys
import time

import numpy as np
import matplotlib

# ginput() needs an interactive GUI backend; the default (e.g. "Agg") can't
# open a clickable window. Try common interactive backends until one loads.
_INTERACTIVE_BACKENDS = ["QtAgg", "Qt5Agg", "TkAgg", "MacOSX"]
_backend_ok = False
for _backend in _INTERACTIVE_BACKENDS:
    try:
        matplotlib.use(_backend, force=True)
        _backend_ok = True
        break
    except Exception:
        continue
 
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
 
if not _backend_ok:
    print(
        "ERROR: No interactive matplotlib backend is available (tried: "
        + ", ".join(_INTERACTIVE_BACKENDS)
        + ").\n"
        "Install one, e.g.:\n"
        "    pip install PyQt5\n"
        "or, if running in Jupyter, add this in a cell before running:\n"
        "    %matplotlib widget    (requires: pip install ipympl)\n"
        "or:\n"
        "    %matplotlib qt        (requires: pip install PyQt5)"
    )
    sys.exit(1)

try:
    import rasterio
    from rasterio.transform import xy as rasterio_xy
except ImportError:
    print("This script requires rasterio. Install with: pip install rasterio")
    sys.exit(1)

try:
    from pyproj import Transformer, CRS
except ImportError:
    print("This script requires pyproj. Install with: pip install pyproj")
    sys.exit(1)


def select_points_on_image(image_array, title, n_points):
    """Show an image and let the user click n_points points on it.
 
    Unlike plt.ginput(), this ignores clicks made while the toolbar's
    zoom or pan tool is active, and ignores non-left-clicks, so you can
    freely zoom/pan to line up a precise click without it being recorded
    as a point.
 
    Returns a list of (x, y) tuples in pixel/data coordinates, in click order.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image_array)
    ax.set_title(
        f"{title}\nLeft-click {n_points} points in order.\n"
        "Zoom/pan freely with the toolbar; those clicks are ignored."
    )
    plt.axis("on")
 
    points = []
    markers = []
 
    def on_click(event):
        # Ignore clicks made while the zoom or pan tool is active.
        toolbar = fig.canvas.toolbar
        if toolbar is not None and toolbar.mode != "":
            return
        # Ignore clicks outside the image axes or that aren't a left-click.
        if event.inaxes != ax or event.button != 1:
            return
 
        points.append((event.xdata, event.ydata))
        marker, = ax.plot(event.xdata, event.ydata, "r+", markersize=12, markeredgewidth=2)
        markers.append(marker)
        ax.set_title(f"{title}\nPoint {len(points)}/{n_points} selected.")
        fig.canvas.draw_idle()
 
    cid = fig.canvas.mpl_connect("button_press_event", on_click)
 
    fig.canvas.draw()
    plt.show(block=False)
 
    # Manually pump the GUI event queue instead of using
    # fig.canvas.start_event_loop(). Jupyter/IPython installs its own GUI
    # event-loop integration for interactive backends, which can make
    # start_event_loop() (and plt.ginput(), which uses it internally) return
    # immediately instead of actually waiting. Polling with flush_events()
    # sidesteps that and works the same in a notebook, a plain script, or
    # an IDE console.
    try:
        while len(points) < n_points and plt.fignum_exists(fig.number):
            fig.canvas.flush_events()
            time.sleep(0.02)
    finally:
        fig.canvas.mpl_disconnect(cid)
        if plt.fignum_exists(fig.number):
            plt.close(fig)
 
    if len(points) < n_points:
        print(f"Warning: only {len(points)} of {n_points} points were selected on {title}.")
    return points
 
 
def load_orthophoto_for_display(ortho_path):
    """Read an orthophoto with rasterio and return an array matplotlib can display."""
    with rasterio.open(ortho_path) as src:
        arr = src.read()  # shape: (bands, rows, cols)
        if arr.shape[0] >= 3:
            img = np.transpose(arr[:3], (1, 2, 0))
        else:
            img = arr[0]
 
        if img.dtype != np.uint8:
            img = img.astype(np.float32)
            img -= img.min()
            if img.max() > 0:
                img = img / img.max()
        return img
 
 
def ortho_pixels_to_wgs84(ortho_path, pixel_points):
    """Convert (col, row) pixel points on the orthophoto to (lon, lat) in WGS84."""
    with rasterio.open(ortho_path) as src:
        transform = src.transform
        src_crs = src.crs
        if src_crs is None:
            raise ValueError(
                "Orthophoto has no CRS information; cannot convert to WGS84."
            )
 
        native_coords = []
        for (col, row) in pixel_points:
            x, y = rasterio_xy(transform, row, col, offset="center")
            native_coords.append((x, y))
 
        target_crs = CRS.from_epsg(4326)  # WGS84
        transformer = Transformer.from_crs(src_crs, target_crs, always_xy=True)
        wgs84_coords = [transformer.transform(x, y) for (x, y) in native_coords]
        return wgs84_coords  # list of (lon, lat)


print("Step 1: click on the ORTHOPHOTO.")
ortho_img = load_orthophoto_for_display(ORTHOPHOTO_PATH)
ortho_pixel_points = select_points_on_image(ortho_img, "Orthophoto", NB_REF_POINTS)

print("Step 2: click on the PNG image (different angle), in the same order.")
png_img = Image.open(FRAME_PATH)
png_array = np.array(png_img)
png_pixel_points = select_points_on_image(png_array, "PNG Photo", NB_REF_POINTS)

if len(ortho_pixel_points) < NB_REF_POINTS or len(png_pixel_points) < NB_REF_POINTS:
    print("Not enough points selected on one or both images. Aborting without writing output.")
    sys.exit(1)

print("Converting orthophoto pixel coordinates to WGS84...")
wgs84_points = ortho_pixels_to_wgs84(ORTHOPHOTO_PATH, ortho_pixel_points)

with open(OUTPUT_TXT_PATH, "w") as f:
    f.write("Point\tOrtho_Lon(WGS84)\tOrtho_Lat(WGS84)\tPNG_Pixel_X\tPNG_Pixel_Y\n")
    for i in range(NB_REF_POINTS):
        lon, lat = wgs84_points[i]
        px, py = png_pixel_points[i]
        f.write(f"{i + 1}\t{lon:.8f}\t{lat:.8f}\t{px:.2f}\t{py:.2f}\n")

print(f"Done. Results written to {OUTPUT_TXT_PATH}")

Step 1: click on the ORTHOPHOTO.
Step 2: click on the PNG image (different angle), in the same order.
Converting orthophoto pixel coordinates to WGS84...
Done. Results written to ../../output/ref_points.txt


In [12]:
# NEW CODE - HOMOGRAPHY

TRACKING_CSV_PATH = TRACKING_RUNS_PATH / "lkpg_full_bbox_centers_xyxy_yolo12s_part4.csv"
OUTPUT_CSV_PATH = HOMOGRAPHY_RUNS_PATH / "output_homography_yolo12s_part4.csv"
REF_POINTS_PATH = Path("../../output/ref_points.txt")
POINT_IN_BBOX = (0.15, 0.86) # proportion of bbox width/height to use to get a point (x, y) from the bbox ((0, 0) is top-left, (1, 1) is bottom-right)


"""
add_geo_coords.py

Adds longitude/latitude columns to a CSV of pixel coordinates, using a
homography (planar projective transform) computed from a small set of
reference points that link pixel coordinates (from a video frame / ortho
image) to real-world WGS84 longitude/latitude.

Usage
-----
    python add_geo_coords.py INPUT.csv REF_POINTS.txt OUTPUT.csv \
        [--x-col x_center] [--y-col y_center] \
        [--lon-col longitude] [--lat-col latitude]

Reference points file format (tab- or whitespace-separated, with header),
matching ref_points.txt:

    Point  Ortho_Lon(WGS84)  Ortho_Lat(WGS84)  PNG_Pixel_X  PNG_Pixel_Y
    1      15.62333972       58.41719991       1410.99      1873.41
    ...

Notes
-----
- A homography needs at least 4 non-collinear point correspondences.
  With more than 4 (as here, 5), OpenCV's findHomography fits a
  least-squares solution (method=0), which also averages out small
  measurement errors in the reference points.
- This assumes pixel coordinates in the input CSV are in the SAME pixel
  space as PNG_Pixel_X / PNG_Pixel_Y in the reference file (i.e. the
  video frame and the ortho/reference PNG share the same resolution and
  alignment). If your video frames have a different resolution than the
  reference PNG, rescale x/y before applying the homography.
"""

import argparse
import sys

import numpy as np
import pandas as pd
import cv2


def load_ref_points(path):
    """Load reference points file into pixel and lon/lat arrays."""
    df = pd.read_csv(path, sep=r"\s+", engine="python")

    required = {"Ortho_Lon(WGS84)", "Ortho_Lat(WGS84)", "PNG_Pixel_X", "PNG_Pixel_Y"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Reference points file is missing expected columns: {missing}\n"
            f"Found columns: {list(df.columns)}"
        )

    pixel_pts = df[["PNG_Pixel_X", "PNG_Pixel_Y"]].to_numpy(dtype=np.float64)
    geo_pts = df[["Ortho_Lon(WGS84)", "Ortho_Lat(WGS84)"]].to_numpy(dtype=np.float64)
    return pixel_pts, geo_pts


def compute_homography(pixel_pts, geo_pts):
    """Fit a homography mapping pixel (x, y) -> (lon, lat)."""
    if len(pixel_pts) < 4:
        raise ValueError(
            f"Need at least 4 reference points to compute a homography, "
            f"got {len(pixel_pts)}."
        )

    # method=0 => regular least-squares fit using ALL points (good when we
    # have more than the minimal 4 points and trust them all).
    H, mask = cv2.findHomography(pixel_pts, geo_pts, method=0)

    if H is None:
        raise RuntimeError("Homography computation failed (degenerate points?).")

    return H


def apply_homography(H, xy):
    """Apply homography H to an (N, 2) array of pixel coords -> (N, 2) lon/lat."""
    n = xy.shape[0]
    homogeneous = np.hstack([xy, np.ones((n, 1))])  # (N, 3)
    transformed = homogeneous @ H.T  # (N, 3)
    w = transformed[:, 2:3]
    # Avoid division by zero just in case
    w[w == 0] = 1e-12
    lonlat = transformed[:, :2] / w
    return lonlat


def report_residuals(H, pixel_pts, geo_pts):
    """Print how well the homography reproduces the reference points."""
    predicted = apply_homography(H, pixel_pts)
    errors = predicted - geo_pts
    # Rough conversion of degree error to meters for a sanity check
    # (1 deg lat ~ 111320 m; 1 deg lon ~ 111320*cos(lat) m)
    lat0 = np.mean(geo_pts[:, 1])
    m_per_deg_lat = 111320.0
    m_per_deg_lon = 111320.0 * np.cos(np.radians(lat0))

    print("Reference point fit residuals (predicted - actual):")
    for i, (e_lon, e_lat) in enumerate(errors, start=1):
        err_m = np.hypot(e_lon * m_per_deg_lon, e_lat * m_per_deg_lat)
        print(f"  Point {i}: dLon={e_lon:+.8f}, dLat={e_lat:+.8f}  (~{err_m:.2f} m)")
    print()


pixel_pts, geo_pts = load_ref_points(REF_POINTS_PATH)
H = compute_homography(pixel_pts, geo_pts)
report_residuals(H, pixel_pts, geo_pts)

data = pd.read_csv(TRACKING_CSV_PATH)

for col in ("x1", "y1", "x2", "y2"):
    if col not in data.columns:
        raise ValueError(
            f"Column '{col}' not found in input CSV. "
            f"Available columns: {list(data.columns)}"
        )

bbox_x, bbox_y = POINT_IN_BBOX
if not (0 <= bbox_x <= 1 and 0 <= bbox_y <= 1):
    raise ValueError("POINT_IN_BBOX values must be between 0 and 1.")

xy = np.column_stack((
    data["x1"] + bbox_x * (data["x2"] - data["x1"]),
    data["y1"] + bbox_y * (data["y2"] - data["y1"]),
)).astype(np.float64)
lonlat = apply_homography(H, xy)

data["longitude"] = lonlat[:, 0]
data["latitude"] = lonlat[:, 1]

data.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Wrote {len(data)} rows with longitude and latitude columns to:")
print(f"  {OUTPUT_CSV_PATH}")


Reference point fit residuals (predicted - actual):
  Point 1: dLon=+0.00000013, dLat=+0.00000238  (~0.26 m)
  Point 2: dLon=-0.00000092, dLat=+0.00000050  (~0.08 m)
  Point 3: dLon=+0.00000092, dLat=-0.00000044  (~0.07 m)
  Point 4: dLon=+0.00000009, dLat=+0.00000093  (~0.10 m)
  Point 5: dLon=-0.00000030, dLat=+0.00000009  (~0.02 m)

Wrote 5944 rows with longitude and latitude columns to:
  modules\Homography\runs\track_lkpg\output_homography_yolo12s_part4.csv


In [4]:
# MILY'S CODE - HOMOGRAPHY

# ---------- Ground control points ----------
# Image points: pixel coordinates [x, y]
'''
image_points = np.array([
    [376.96, 901.18],
    [604.11, 352.48],
    [1375.22, 316.35],
    [1297.00, 811.88]
], dtype=np.float32)

world_points = np.array([
    [59.56892476200374, 17.53222047512886],
    [59.56877228708751, 17.53130739317016],
    [59.56923788112685, 17.53050557603159],
    [59.56944760043758, 17.5311971431628]
], dtype=np.float32)
'''
TRACKING_CSV_PATH = TRACKING_RUNS_PATH / "lkpg_full_bbox_centers_xyxy_yolo12s_part1.csv"
OUTPUT_CSV_PATH = HOMOGRAPHY_RUNS_PATH / "output_homography_yolo12s_part1.csv"

# ---------- Load reference points from file ----------
ref_points_path = Path("../../output/ref_points.txt")
ref_points = pd.read_csv(ref_points_path, sep="\t")
image_points = ref_points[["PNG_Pixel_X", "PNG_Pixel_Y"]].to_numpy(dtype=np.float32)
world_points = ref_points[["Ortho_Lat(WGS84)", "Ortho_Lon(WGS84)"]].to_numpy(dtype=np.float32)

# ---------- Compute homography ----------
H, status = cv2.findHomography(image_points, world_points)

# ---------- Load CSV ----------
df = pd.read_csv(TRACKING_CSV_PATH)

# ---------- Extract pixel center coordinates ----------
pixels = df[['x_center', 'y_center']].values.astype(np.float32)
pixels = pixels.reshape(-1, 1, 2)

# ---------- Transform pixel coordinates to GPS ----------
latlon = cv2.perspectiveTransform(pixels, H)

# Since world_points are in [latitude, longitude] format
df['latitude'] = latlon[:, 0, 0]
df['longitude'] = latlon[:, 0, 1]

# ---------- Save output CSV ----------
df.to_csv(OUTPUT_CSV_PATH, index=False)

print("Homography conversion complete.")
print(f"Output saved to: {OUTPUT_CSV_PATH}")

Homography conversion complete.
Output saved to: modules\Homography\runs\track_lkpg\output_homography_yolo12s_part1.csv
